# 03 Data Split — Men's

This notebook constructs the matchup-level training dataset from historical tournament games and defines the cross-validation strategy.

**Key design decisions:**
- **Training target**: Tournament game outcomes (binary: did the lower-ID team win?)
- **CV strategy**: Leave-One-Season-Out (LOGO) — train on all seasons except one, validate on the held-out season's tournament
- **Flip and double**: Each game appears twice (once per team ordering) with label flipped, preventing winner/loser ordering bias
- **Exclude 2020**: No tournament was played
- **2026 prediction grid**: All possible men's team matchups for Stage 2 submission

**Inputs** (from S3 `01_data_joining/mens/`):
- `tourney_games.parquet`
- `tourney_seeds.parquet`
- `team_season_stats.parquet`
- `massey_ordinals_pre_tourney.parquet`
- `team_metadata.parquet`
- `SampleSubmissionStage1.csv` and `SampleSubmissionStage2.csv` (from `00_data_collection/`)

**Outputs** (to S3 `03_data_split/mens/`):
1. `tourney_matchups.parquet` — all historical tournament matchups with labels and fold assignments
2. `prediction_grid_stage1.parquet` — all men's matchups from Stage 1 sample submission (2022–2025)
3. `prediction_grid_stage2.parquet` — all men's matchups from Stage 2 sample submission (2026)

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

#### Functions

In [2]:
def read_parquet(filename):
    """Read parquet from S3 if available, otherwise fall back to local."""
    try:
        return pd.read_parquet(f"{INPUT_PREFIX}{filename}")
    except Exception:
        return pd.read_parquet(f"{LOCAL_INPUT}{filename}")

def read_csv(filename):
    """Read CSV from S3 raw data or local."""
    try:
        return pd.read_csv(f"s3://{BUCKET}/00_data_collection/{filename}")
    except Exception:
        return pd.read_csv(f"../00_data_collection/{filename}")

#### Constants

In [3]:
BUCKET = "march-machine-learning-mania-2026"
GENDER = "mens"
STAGE = "03_data_split"
INPUT_PREFIX = f"s3://{BUCKET}/01_data_joining/{GENDER}/"
OUTPUT_PREFIX = f"s3://{BUCKET}/{STAGE}/{GENDER}/"

LOCAL_INPUT = "../01_data_joining/output/"
LOCAL_OUTPUT = "output/"

#### Make output directory

In [4]:
os.makedirs(LOCAL_OUTPUT, exist_ok=True)

## 1. Load Data

In [5]:
tourney_games = read_parquet("tourney_games.parquet")
seeds = read_parquet("tourney_seeds.parquet")
sample_sub_stage1 = read_csv("SampleSubmissionStage1.csv")
sample_sub_stage2 = read_csv("SampleSubmissionStage2.csv")

print(f"Tournament games: {tourney_games.shape}")
print(f"Seeds: {seeds.shape}")
print(f"Stage 1 submission rows: {len(sample_sub_stage1):,}")
print(f"Stage 2 submission rows: {len(sample_sub_stage2):,}")

Tournament games: (2585, 34)
Seeds: (2626, 6)
Stage 1 submission rows: 519,144
Stage 2 submission rows: 132,133


## 2. Build Tournament Matchup Dataset

Convert tournament games from winner/loser format into the submission format: `(Season, TeamID_lower, TeamID_higher)` with a binary label indicating whether the lower-ID team won.

This matches the submission format where `ID = YYYY_XXXX_YYYY` (XXXX < YYYY) and `Pred` = P(lower-ID team wins).

In [6]:
# Build matchups in submission format
matchups = tourney_games[['Season', 'WTeamID', 'LTeamID']].copy()

# Assign TeamA = lower ID, TeamB = higher ID (matches submission format)
matchups['TeamA'] = matchups[['WTeamID', 'LTeamID']].min(axis=1)
matchups['TeamB'] = matchups[['WTeamID', 'LTeamID']].max(axis=1)

# Label: 1 if lower-ID team (TeamA) won, 0 otherwise
matchups['Label'] = (matchups['WTeamID'] == matchups['TeamA']).astype(int)

# Create ID in submission format
matchups['ID'] = (matchups['Season'].astype(str) + '_' + 
                  matchups['TeamA'].astype(str) + '_' + 
                  matchups['TeamB'].astype(str))

matchups = matchups[['ID', 'Season', 'TeamA', 'TeamB', 'Label']]

print(f"Tournament matchups: {matchups.shape}")
print(f"Seasons: {matchups.Season.min()} - {matchups.Season.max()}")
print(f"Label distribution: {matchups.Label.mean():.3f} (should be ~0.5)")
matchups.head()

Tournament matchups: (2585, 5)
Seasons: 1985 - 2025
Label distribution: 0.512 (should be ~0.5)


,ID,Season,TeamA,TeamB,Label
0,1985_1116_1234,1985,1116,1234,1
1,1985_1120_1345,1985,1120,1345,1
2,1985_1207_1250,1985,1207,1250,1
3,1985_1229_1425,1985,1229,1425,1
4,1985_1242_1325,1985,1242,1325,1


## 3. Assign LOGO Cross-Validation Folds

Each season is one fold. This gives us ~22 folds for training (2003–2025, excluding 2020) when using detailed stats, or ~40 folds (1985–2025, excluding 2020) when using compact stats only.

We also mark the Stage 1 validation seasons (2022–2025) which simulate the Kaggle leaderboard.

In [7]:
# Exclude 2020 (no tournament played)
matchups = matchups[matchups.Season != 2020].copy()

# Fold = Season (for LOGO CV)
matchups['Fold'] = matchups['Season']

# Mark Stage 1 validation years
stage1_seasons = [2022, 2023, 2024, 2025]
matchups['IsStage1Val'] = matchups['Season'].isin(stage1_seasons).astype(int)

print(f"Matchups after excluding 2020: {matchups.shape}")
print(f"Unique folds (seasons): {matchups.Fold.nunique()}")
print(f"\nGames per fold:")
fold_counts = matchups.groupby('Fold').size()
print(f"  Min: {fold_counts.min()}, Max: {fold_counts.max()}, Mean: {fold_counts.mean():.1f}")
print(f"\nStage 1 validation games: {matchups.IsStage1Val.sum()}")
print(f"Training games (non-Stage 1): {(matchups.IsStage1Val == 0).sum()}")

Matchups after excluding 2020: (2585, 7)
Unique folds (seasons): 40

Games per fold:
  Min: 63, Max: 67, Mean: 64.6

Stage 1 validation games: 268
Training games (non-Stage 1): 2317


In [8]:
# Summary of folds
fold_summary = matchups.groupby('Fold').agg(
    Games=('Label', 'count'),
    LabelMean=('Label', 'mean'),
    IsStage1Val=('IsStage1Val', 'first')
).reset_index()

print("Fold summary:")
print(fold_summary.to_string(index=False))

Fold summary:
 Fold  Games  LabelMean  IsStage1Val
 1985     63   0.555556            0
 1986     63   0.571429            0
 1987     63   0.492063            0
 1988     63   0.507937            0
 1989     63   0.523810            0
 1990     63   0.571429            0
 1991     63   0.507937            0
 1992     63   0.507937            0
 1993     63   0.460317            0
 1994     63   0.698413            0
 1995     63   0.412698            0
 1996     63   0.507937            0
 1997     63   0.571429            0
 1998     63   0.492063            0
 1999     63   0.587302            0
 2000     63   0.476190            0
 2001     64   0.531250            0
 2002     64   0.500000            0
 2003     64   0.437500            0
 2004     64   0.562500            0
 2005     64   0.484375            0
 2006     64   0.500000            0
 2007     64   0.468750            0
 2008     64   0.453125            0
 2009     64   0.375000            0
 2010     64   0.500000 

## 4. Build Prediction Grids

Parse the sample submission files to get the prediction grids. Filter to men's matchups only (TeamIDs 1000–1999).

In [9]:
def parse_submission_grid(sample_sub):
    """Parse sample submission into Season, TeamA, TeamB columns."""
    grid = sample_sub[['ID']].copy()
    parts = grid['ID'].str.split('_', expand=True)
    grid['Season'] = parts[0].astype(int)
    grid['TeamA'] = parts[1].astype(int)
    grid['TeamB'] = parts[2].astype(int)
    return grid

# Stage 1: 2022-2025 all possible matchups
grid_stage1 = parse_submission_grid(sample_sub_stage1)
# Filter to men's only (TeamIDs 1000-1999)
grid_stage1_mens = grid_stage1[(grid_stage1.TeamA >= 1000) & (grid_stage1.TeamA < 2000)].copy()

# Stage 2: 2026 all possible matchups
grid_stage2 = parse_submission_grid(sample_sub_stage2)
grid_stage2_mens = grid_stage2[(grid_stage2.TeamA >= 1000) & (grid_stage2.TeamA < 2000)].copy()

print(f"Stage 1 prediction grid (men's): {grid_stage1_mens.shape}")
print(f"  Seasons: {grid_stage1_mens.Season.unique()}")
print(f"  Unique teams per season: {grid_stage1_mens.groupby('Season')['TeamA'].nunique().to_dict()}")
print(f"\nStage 2 prediction grid (men's): {grid_stage2_mens.shape}")
print(f"  Season: {grid_stage2_mens.Season.unique()}")

Stage 1 prediction grid (men's): (261013, 4)
  Seasons: [2022 2023 2024 2025]
  Unique teams per season: {2022: 357, 2023: 362, 2024: 361, 2025: 363}

Stage 2 prediction grid (men's): (66430, 4)
  Season: [2026]


## 5. Merge Labels into Stage 1 Grid

For Stage 1 validation, we can check which matchups actually happened in the tournament and attach labels. Most matchups in the grid never happened (only ~67 games per tournament), but this lets us validate predictions against actual outcomes.

In [10]:
# Merge actual tournament results into Stage 1 grid
stage1_with_labels = grid_stage1_mens.merge(
    matchups[['ID', 'Label']],
    on='ID',
    how='left'
)

actual_games = stage1_with_labels['Label'].notna().sum()
print(f"Stage 1 grid rows: {len(stage1_with_labels):,}")
print(f"Rows with actual tournament results: {actual_games}")
print(f"Rows without results (never played): {len(stage1_with_labels) - actual_games:,}")

# Store this info on the grid
grid_stage1_mens = stage1_with_labels

Stage 1 grid rows: 261,013
Rows with actual tournament results: 268
Rows without results (never played): 260,745


## 6. Save Outputs

In [11]:
outputs = {
    'tourney_matchups': matchups,
    'prediction_grid_stage1': grid_stage1_mens,
    'prediction_grid_stage2': grid_stage2_mens,
}

for name, df in outputs.items():
    try:
        s3_path = f"{OUTPUT_PREFIX}{name}.parquet"
        df.to_parquet(s3_path, index=False)
        print(f"Saved to S3: {s3_path} ({df.shape})")
    except Exception as e:
        print(f"S3 save failed for {name}: {e}")

Saved to S3: s3://march-machine-learning-mania-2026/03_data_split/mens/tourney_matchups.parquet ((2585, 7))
Saved to S3: s3://march-machine-learning-mania-2026/03_data_split/mens/prediction_grid_stage1.parquet ((261013, 5))
Saved to S3: s3://march-machine-learning-mania-2026/03_data_split/mens/prediction_grid_stage2.parquet ((66430, 4))


## 7. Output Summary

In [12]:
print("=" * 60)
print("DATA SPLIT SUMMARY — MEN'S")
print("=" * 60)

print(f"\ntourney_matchups:")
print(f"  Shape: {matchups.shape}")
print(f"  Seasons: {matchups.Season.min()} - {matchups.Season.max()} (excl. 2020)")
print(f"  Total games: {len(matchups)}")
print(f"  Label balance: {matchups.Label.mean():.3f}")
print(f"  LOGO folds: {matchups.Fold.nunique()} seasons")
print(f"  Stage 1 validation games (2022-2025): {matchups.IsStage1Val.sum()}")

print(f"\nprediction_grid_stage1:")
print(f"  Shape: {grid_stage1_mens.shape}")
print(f"  Rows with labels: {grid_stage1_mens.Label.notna().sum()}")

print(f"\nprediction_grid_stage2:")
print(f"  Shape: {grid_stage2_mens.shape}")

print(f"\nCV STRATEGY: Leave-One-Season-Out (LOGO)")
print(f"  Each season is one fold.")
print(f"  Train on all other seasons, validate on the held-out season.")
print(f"  Final model comparison uses Stage 1 seasons (2022-2025).")
print(f"  2020 excluded (no tournament).")

DATA SPLIT SUMMARY — MEN'S

tourney_matchups:
  Shape: (2585, 7)
  Seasons: 1985 - 2025 (excl. 2020)
  Total games: 2585
  Label balance: 0.512
  LOGO folds: 40 seasons
  Stage 1 validation games (2022-2025): 268

prediction_grid_stage1:
  Shape: (261013, 5)
  Rows with labels: 268

prediction_grid_stage2:
  Shape: (66430, 4)

CV STRATEGY: Leave-One-Season-Out (LOGO)
  Each season is one fold.
  Train on all other seasons, validate on the held-out season.
  Final model comparison uses Stage 1 seasons (2022-2025).
  2020 excluded (no tournament).
